# Text Splitter — 청킹 (Chunking)


### 왜 분할이 필요한가?

| 문제 | 해결 |
|------|------|
| LLM 컨텍스트 윈도우 제한 | 작은 청크로 분할 |
| 긴 문서에서 정확한 검색 어려움 | 관련 청크만 검색 |
| 토큰 비용 | 효율적인 토큰 사용 |


### 참고자료
- https://docs.langchain.com/oss/python/integrations/splitters
- https://wikidocs.net/231430

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [1]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-community langchain-text-splitters langchain-openai langchain-experimental pypdfium2 pypdf

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [2]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. 데모용 긴 문서

In [3]:
SAMPLE_DOC = """
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 장비 수령, 계정 활성화, 부서별 업무 소개가 진행된다. 노트북과 출입카드는 입사 당일 지급되며, 사내 시스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.

### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.

### 3. 법인카드 및 경비 처리
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.

### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.
"""

print(f"문서 길이: {len(SAMPLE_DOC)} 자")

문서 길이: 971 자


## 3. `RecursiveCharacterTextSplitter`, 가장 많이 쓰는 분할기

In [4]:
# RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,             # 한 청크 목표 크기 (자)
    chunk_overlap=60,           # 겹치는 글자 수 (맥락 보존)
    separators=['\n\n', '\n', '다.', ' ', '']     # 분할 우선순위
)

chunks = splitter.split_text(SAMPLE_DOC)
print(f'청크 수: {len(chunks)}')
for i, c in enumerate(chunks):
    print(f'---청크 {i} ({len(c)}자) ---')
    print((c[:150]))
    print()

청크 수: 4
---청크 0 (244자) ---
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션

---청크 1 (241자) ---
### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에

---청크 2 (238자) ---
### 3. 법인카드 및 경비 처리
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의

---청크 3 (240자) ---
### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야



## 4. overlap 효과,
- 왜 80자를 겹치게 두나?
- 청크 경계에서 문장이 끊기면 의미가 잘림
- 인접 청크끼리 일부를 겹치게 두면 검색 시 한 청크가 잡혀도 양쪽 맥락이 살아남

### overlap 0 일 때

In [5]:
splitter_no_overlap = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=0,
)

chunks_no_ovl = splitter_no_overlap.split_text(SAMPLE_DOC)

# 각 청크의 마지막 / 다음 청크의 처음 비교
# overlap이 없으면 앞 청크의 끝과 다음 청크의 시작이 이어지지만, 같은 내용 반복x
for i in range(len(chunks_no_ovl)-1):
    end = chunks_no_ovl[i][-30:]
    start = chunks_no_ovl[i+1][30:]
    print(f'청크{i} 끝 : ...{end!r}')
    print(f'청크{i+1} 시작 : {start!r}...')
    print()

청크0 끝 : ...'스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.'
청크1 시작 : ' 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.'...

청크1 끝 : ...'무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.'
청크2 시작 : '은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.'...

청크2 끝 : ...'첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.'
청크3 시작 : '문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.'...



In [6]:
# 각 청크의 마지막 / 다음 청크의 처음 비교
for i in range(len(chunks_no_ovl) - 1):
    end = chunks_no_ovl[i][-30:]
    start = chunks_no_ovl[i + 1][:30]
    print(f"청크{i} 끝: ...{end!r}")
    print(f"청크{i+1} 시작: {start!r}...")
    print()

청크0 끝: ...'스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.'
청크1 시작: '### 2. 재택근무 신청\n재택근무는 최소 하루 전까지'...

청크1 끝: ...'무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.'
청크2 시작: '### 3. 법인카드 및 경비 처리\n법인카드 사용 내역'...

청크2 끝: ...'첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.'
청크3 시작: '### 4. 보안 및 개인정보 보호\n개인정보가 포함된 '...



## 5. Token 기준 분할

- LLM 컨텍스트는 "자" 가 아니라 "토큰" 으로 제한됨
- 토큰 기준으로 자르려면:

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name='text-embedding-3-small',
    chunk_size=200,
    chunk_overlap=40
)

In [8]:

token_chunks = token_splitter.split_text(SAMPLE_DOC)
print(f"토큰 기준 청크 수: {len(token_chunks)}")
for i, c in enumerate(token_chunks):
    print(f"--- 청크 {i} ({len(c)}자) ---")
    print(c[:120])
    print()

토큰 기준 청크 수: 10
--- 청크 0 (15자) ---
## 회사 업무 운영 가이드

--- 청크 1 (17자) ---
### 1. 신규 입사자 온보딩

--- 청크 2 (209자) ---
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 

--- 청크 3 (14자) ---
### 2. 재택근무 신청

--- 청크 4 (190자) ---
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근

--- 청크 5 (70자) ---
10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.

--- 청크 6 (19자) ---
### 3. 법인카드 및 경비 처리

--- 청크 7 (218자) ---
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 

--- 청크 8 (19자) ---
### 4. 보안 및 개인정보 보호

--- 청크 9 (220자) ---
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공



## 6. 청크 크기 trade-off

| 청크 크기 | 장점 | 단점 |
|---|---|---|
| 작음 (200자) | 검색 정밀도 ↑, 토큰 비용 ↓ | 맥락 부족, 청크 수 ↑ |
| 중간 (500~800자) | 균형 | (없음) |
| 큼 (1500자+) | 풍부한 맥락 | 검색 정밀도 ↓, 비용 ↑ |

실무 시작점은 문서 성격에 따라 다르지만, 일반 문서 RAG 는 보통 300 ~ 800 토큰, overlap 10 ~ 20% 정도에서 실험을 시작합니다.


## 7. 정리

- 긴 문서는 분할 필수
- `RecursiveCharacterTextSplitter` 가 기본
- `chunk_overlap` 으로 경계 손실 방지
- 토큰 기준 분할은 `from_tiktoken_encoder`


## [실습]
1. `chunk_size` 를 100 / 800 / 2000 으로 바꿔 청크 수 변화.
2. `separators` 를 한국어 친화로 (`["\n\n", "\n", ". ", "다. ", " "]`) 바꿔 분할 결과 비교.
3. PDF (`PyPDFLoader` 사용) 를 로드해 분할.
4. 마크다운 헤더 기준 분할 (`MarkdownHeaderTextSplitter`) 시도, 청크가 의미 단위로 떨어지는지.

In [9]:
import pandas as pd

chunk_sizes = [100, 800, 2000]

rows = []

for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=80,
        separators=["\n\n", "\n", "다. ", ". ", " ", ""],
    )

    chunks = splitter.split_text(SAMPLE_DOC)

    rows.append({
        "chunk_size": size,
        "chunk_overlap": 80,
        "chunk_count": len(chunks),
        "min_len": min(len(c) for c in chunks),
        "max_len": max(len(c) for c in chunks),
        "avg_len": round(sum(len(c) for c in chunks) / len(chunks), 1),
    })

pd.DataFrame(rows)

,chunk_size,chunk_overlap,chunk_count,min_len,max_len,avg_len
0,100,80,18,14,99,55.3
1,800,80,2,240,727,483.5
2,2000,80,1,969,969,969.0


In [10]:
# 각 chunk_size별 첫 3개 청크 미리보기
for size in chunk_sizes:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=80,
        separators=["\n\n", "\n", "다. ", ". ", " ", ""],
    )

    chunks = splitter.split_text(SAMPLE_DOC)

    print("=" * 80)
    print(f"chunk_size={size}, 청크 수={len(chunks)}")
    print("=" * 80)

    for i, chunk in enumerate(chunks[:3]):
        print(f"\n--- 청크 {i} ({len(chunk)}자) ---")
        print(chunk[:300])

chunk_size=100, 청크 수=18

--- 청크 0 (15자) ---
## 회사 업무 운영 가이드

--- 청크 1 (17자) ---
### 1. 신규 입사자 온보딩

--- 청크 2 (93자) ---
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있
chunk_size=800, 청크 수=2

--- 청크 0 (727자) ---
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 장비 수령, 계정 활성화, 부서별 업무 소개가 진행된다. 노트북과 출입카드는 입사 당일 지급되며, 사내 시스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.

### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청

--- 청크 1 (240자) ---
### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 공유 링크를 사용해야 하며, 개인 이메일이나 개인 클라우드로 파일을 전송해서는 안 된다. 보안 사고가 의심되는 경우 즉시 보안팀에 신고하고, 관련 로그와 발생 시각을 기록해야 한다.
chunk_size=2000, 청크 수=1

--- 청크 0 (969자) ---
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 

In [11]:
separator_cases = {
    "기본형": ["\n\n", "\n", ".", " ", ""],
    "한국어 친화형": ["\n\n", "\n", "다. ", ". ", " ", ""],
}

rows = []

for name, separators in separator_cases.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80,
        separators=separators,
    )

    chunks = splitter.split_text(SAMPLE_DOC)

    rows.append({
        "case": name,
        "separators": separators,
        "chunk_count": len(chunks),
        "min_len": min(len(c) for c in chunks),
        "max_len": max(len(c) for c in chunks),
        "avg_len": round(sum(len(c) for c in chunks) / len(chunks), 1),
    })

pd.DataFrame(rows)

,case,separators,chunk_count,min_len,max_len,avg_len
0,기본형,"[\n\n, \n, ., , ]",4,238,244,240.8
1,한국어 친화형,"[\n\n, \n, 다. , . , , ]",4,238,244,240.8


In [12]:
# 분할 결과 눈으로 비교
for name, separators in separator_cases.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=400,
        chunk_overlap=80,
        separators=separators,
    )

    chunks = splitter.split_text(SAMPLE_DOC)

    print("=" * 80)
    print(name)
    print(f"separators={separators}")
    print(f"청크 수={len(chunks)}")
    print("=" * 80)

    for i, chunk in enumerate(chunks[:5]):
        print(f"\n--- 청크 {i} ({len(chunk)}자) ---")
        print(chunk[:250])

기본형
separators=['\n\n', '\n', '.', ' ', '']
청크 수=4

--- 청크 0 (244자) ---
## 회사 업무 운영 가이드

### 1. 신규 입사자 온보딩
신규 입사자는 입사 후 7일 이내에 필수 보안 교육을 이수해야 한다. 보안 교육을 완료하지 않으면 사내 메일, 협업 도구, 문서 관리 시스템 접근 권한이 제한될 수 있다. 입사 첫 주에는 인사팀 오리엔테이션, IT 장비 수령, 계정 활성화, 부서별 업무 소개가 진행된다. 노트북과 출입카드는 입사 당일 지급되며, 사내 시스템 비밀번호는 최초 로그인 후 반드시 변경해야 한다.

--- 청크 1 (241자) ---
### 2. 재택근무 신청
재택근무는 최소 하루 전까지 근태 관리 시스템에서 신청해야 한다. 신청 시 근무 장소, 연락 가능한 전화번호, 주요 업무 계획을 입력해야 하며, 팀장의 승인을 받은 뒤 재택근무가 확정된다. 긴급 상황으로 당일 재택근무가 필요한 경우에는 팀장에게 메신저로 먼저 알리고, 당일 오전 10시까지 근태 시스템에 사후 신청을 등록해야 한다. 재택근무 중에는 업무 시간 동안 메신저 상태를 온라인으로 유지해야 한다.

--- 청크 2 (238자) ---
### 3. 법인카드 및 경비 처리
법인카드 사용 내역은 결제일 기준 5영업일 이내에 영수증과 함께 경비 처리 시스템에 등록해야 한다. 회의비를 사용한 경우 참석자 명단과 회의 목적을 함께 입력해야 한다. 개인 물품 구매, 사전 승인 없는 고가 장비 구매, 주류 중심의 회식 비용은 경비 처리 대상에서 제외된다. 영수증을 분실한 경우에는 카드 사용 내역서와 경비 사유서를 첨부해야 하며, 재무팀 검토 후 승인 여부가 결정된다.

--- 청크 3 (240자) ---
### 4. 보안 및 개인정보 보호
개인정보가 포함된 문서는 외부 공유 전에 반드시 비식별 처리해야 한다. 고객명, 전화번호, 이메일, 주소, 주민등록번호, 결제 정보는 마스킹 대상에 포함된다. 외부 협력사와 자료를 공유할 때는 문서 관리 시스템의 